In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vaanyakapur/alice-in-wonderland/alice_in_wonderland.txt
/kaggle/input/datasets/vaanyakapur/pride-and-prejudice/pride_and_prejudice.txt .txt
/kaggle/input/datasets/vaanyakapur/frankenstein/frankenstein.txt .txt
/kaggle/input/datasets/vaanyakapur/the-adventures-of-sherlock-holmes/the_adventures_of_sherlock_holmes.txt


# Narrative Volatility Index using NLP

Great novels rarely stay emotionally flat. In economics, volatility measures how unstable a market signal is over time. This project applies the same idea to literature. Instead of tracking stock prices, this notebook tracks sentiment across a novel.

The main metric is the **Narrative Volatility Index**, or NVI.

- Low NVI: stable emotional tone, calm pacing, exposition, or possible flatness
- High NVI: rapid emotional movement, plot turns, conflict, surprise, or heightened tension

This project uses public-domain novels and lexicon-based sentiment analysis to create a data-driven map of narrative pacing. This project was primarily designed for middle-school writers in the ISB Novel Writing Challenge to identify where their stories may need stronger pacing or emotional variation.

In [2]:
!pip -q install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 1.2 MB/s eta 0:00:00


In [3]:
import os
import re
import glob
import numpy as np
import pandas as pd

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, Markdown

In [4]:
txt_files = glob.glob("/kaggle/input/**/*.txt", recursive=True)

print("Text files found:")
for i, path in enumerate(txt_files):
    print(i, path)

if len(txt_files) == 0:
    raise FileNotFoundError("No .txt files found. Make sure you added your Kaggle dataset to this notebook.")

NOVEL_FILE = txt_files[0]
print("\nUsing this file:")
print(NOVEL_FILE)

Text files found:
0 /kaggle/input/datasets/vaanyakapur/alice-in-wonderland/alice_in_wonderland.txt
1 /kaggle/input/datasets/vaanyakapur/pride-and-prejudice/pride_and_prejudice.txt .txt
2 /kaggle/input/datasets/vaanyakapur/frankenstein/frankenstein.txt .txt
3 /kaggle/input/datasets/vaanyakapur/the-adventures-of-sherlock-holmes/the_adventures_of_sherlock_holmes.txt

Using this file:
/kaggle/input/datasets/vaanyakapur/alice-in-wonderland/alice_in_wonderland.txt


In [5]:
with open(NOVEL_FILE, "r", encoding="utf-8", errors="ignore") as file:
    raw_text = file.read()

print("Original character count:", len(raw_text))


def strip_gutenberg_boilerplate(text):
    """
    Removes common Project Gutenberg header and footer text.
    This keeps the analysis focused on the actual novel.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    start_match = re.search(
        r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if start_match:
        text = text[start_match.end():]

    end_match = re.search(
        r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if end_match:
        text = text[:end_match.start()]

    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


clean_text = strip_gutenberg_boilerplate(raw_text)

print("Cleaned character count:", len(clean_text))
print("\nPreview:")
print(clean_text[:1000])

Original character count: 105045
Cleaned character count: 83288

Preview:
ALICE'S ADVENTURES
 UNDER GROUND

 _BEING A FACSIMILE OF THE_
 _ORIGINAL MS. BOOK_
 _AFTERWARDS DEVELOPED INTO_
 "_ALICE'S ADVENTURES IN WONDERLAND_"

 BY

 LEWIS CARROLL

 _WITH THIRTY-SEVEN ILLUSTRATIONS
 BY THE AUTHOR_

 _PRICE FOUR SHILLINGS_

 London

 MACMILLAN AND CO.
 AND NEW YORK
 1886

 * * * * *

CONTENTS.

CHAPTER

 I. DOWN THE RABBIT-HOLE. THE POOL OF TEARS

 II. A LONG TALE. THE RABBIT SENDS IN A LITTLE BILL

III. ADVICE FROM A CATERPILLAR

 IV. THE QUEEN'S CROQUET-GROUND. THE MOCK TURTLE'S STORY. THE
LOBSTER QUADRILLE. WHO STOLE THE TARTS?

 * * * * *

Chapter 1

[Illustration]

Alice was beginning to get very tired of sitting by her sister on
the bank, and of having nothing to do: once or twice she had
peeped into the book her sister was reading, but it had no
pictures or conversations in it, and where is the use of a book,
thought Alice, without pictures or conversations? So she was
considering i

In [6]:
CHUNK_SIZE = 500
STEP_SIZE = 250

word_tokens = re.findall(r"\S+", clean_text)

print("Approximate word count:", len(word_tokens))

chunks = []

for start in range(0, len(word_tokens) - CHUNK_SIZE + 1, STEP_SIZE):
    chunk_tokens = word_tokens[start:start + CHUNK_SIZE]
    chunk_text = " ".join(chunk_tokens)

    chunks.append({
        "chunk_id": len(chunks) + 1,
        "start_word": start + 1,
        "end_word": start + len(chunk_tokens),
        "progress_pct": round(((start + CHUNK_SIZE / 2) / len(word_tokens)) * 100, 2),
        "text": chunk_text
    })

df = pd.DataFrame(chunks)

print("Number of rolling chunks:", len(df))
df.head()

Approximate word count: 15352
Number of rolling chunks: 60


,chunk_id,start_word,end_word,progress_pct,text
0,1,1,500,1.63,ALICE'S ADVENTURES UNDER GROUND _BEING A FACSI...
1,2,251,750,3.26,"have wondered at this, but at the time it all ..."
2,3,501,1000,4.89,"""Orange Marmalade,"" but to her great disappoin..."
3,4,751,1250,6.51,shall have to ask them what the name of the co...
4,5,1001,1500,8.14,"a heap of sticks and shavings, and the fall wa..."


In [7]:
analyzer = SentimentIntensityAnalyzer()

def chunk_sentiment_score(text):
    """
    Scores a chunk by averaging the VADER compound scores of its sentences.
    This avoids one long chunk becoming too extreme just because it has many words.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text)

    scores = []
    for sentence in sentences:
        if len(sentence.split()) >= 4:
            score = analyzer.polarity_scores(sentence)["compound"]
            scores.append(score)

    if len(scores) == 0:
        return analyzer.polarity_scores(text)["compound"]

    return float(np.mean(scores))


df["sentiment"] = df["text"].apply(chunk_sentiment_score)

df["sentiment_zone"] = pd.cut(
    df["sentiment"],
    bins=[-1, -0.05, 0.05, 1],
    labels=["Negative", "Neutral", "Positive"]
)

df[["chunk_id", "progress_pct", "sentiment", "sentiment_zone"]].head(10)

,chunk_id,progress_pct,sentiment,sentiment_zone
0,1,1.63,0.118188,Positive
1,2,3.26,0.260592,Positive
2,3,4.89,0.211045,Positive
3,4,6.51,0.105683,Positive
4,5,8.14,0.091033,Positive
5,6,9.77,0.234760,Positive
6,7,11.40,-0.101011,Negative
7,8,13.03,-0.099455,Negative
8,9,14.66,0.048861,Neutral
9,10,16.28,0.082813,Positive


In [8]:
VOLATILITY_WINDOW = 5

df["sentiment_smooth"] = df["sentiment"].rolling(
    window=3,
    center=True,
    min_periods=1
).mean()

df["nvi_raw"] = df["sentiment"].rolling(
    window=VOLATILITY_WINDOW,
    center=True,
    min_periods=2
).std()

max_volatility = df["nvi_raw"].max()

if pd.isna(max_volatility) or max_volatility == 0:
    df["nvi"] = 0
else:
    df["nvi"] = (df["nvi_raw"] / max_volatility) * 100

df[["chunk_id", "progress_pct", "sentiment", "nvi"]].head(10)

,chunk_id,progress_pct,sentiment,nvi
0,1,1.63,0.118188,18.147016
1,2,3.26,0.260592,18.702495
2,3,4.89,0.211045,18.677046
3,4,6.51,0.105683,19.401276
4,5,8.14,0.091033,33.364948
5,6,9.77,0.234760,36.374872
6,7,11.40,-0.101011,35.458902
7,8,13.03,-0.099455,35.265150
8,9,14.66,0.048861,28.915673
9,10,16.28,0.082813,23.916179


In [9]:
fig = px.line(
    df,
    x="progress_pct",
    y="sentiment",
    title="Sentiment Arc Across the Novel",
    labels={
        "progress_pct": "Novel Progress (%)",
        "sentiment": "Sentiment Score"
    },
    hover_data=["chunk_id", "start_word", "end_word", "sentiment_zone"]
)

fig.add_hline(y=0, line_dash="dash", annotation_text="Neutral emotional baseline")

fig.update_layout(
    height=500,
    template="plotly_white"
)

fig.show()

In [10]:
top_peaks = df.nlargest(5, "nvi")

fig = px.line(
    df,
    x="progress_pct",
    y="nvi",
    title="Narrative Volatility Index Across the Novel",
    labels={
        "progress_pct": "Novel Progress (%)",
        "nvi": "Narrative Volatility Index"
    },
    hover_data=["chunk_id", "start_word", "end_word", "sentiment"]
)

fig.add_scatter(
    x=top_peaks["progress_pct"],
    y=top_peaks["nvi"],
    mode="markers+text",
    text=["Peak " + str(i + 1) for i in range(len(top_peaks))],
    textposition="top center",
    name="Highest volatility moments"
)

fig.update_layout(
    height=500,
    template="plotly_white"
)

fig.show()

In [11]:
high_threshold = df["nvi"].quantile(0.80)
low_threshold = df["nvi"].quantile(0.20)

df["pacing_zone"] = np.select(
    [
        df["nvi"] >= high_threshold,
        df["nvi"] <= low_threshold
    ],
    [
        "High tension / rapid emotional shifts",
        "Low tension / stable emotional tone"
    ],
    default="Moderate pacing"
)

pacing_summary = df["pacing_zone"].value_counts().reset_index()
pacing_summary.columns = ["Pacing Zone", "Number of Chunks"]

pacing_summary

,Pacing Zone,Number of Chunks
0,Moderate pacing,36
1,Low tension / stable emotional tone,12
2,High tension / rapid emotional shifts,12


In [12]:
diagnostic_table = df[[
    "chunk_id",
    "progress_pct",
    "start_word",
    "end_word",
    "sentiment",
    "nvi",
    "pacing_zone"
]].copy()

diagnostic_table["sentiment"] = diagnostic_table["sentiment"].round(3)
diagnostic_table["nvi"] = diagnostic_table["nvi"].round(1)

diagnostic_table.head(15)

,chunk_id,progress_pct,start_word,end_word,sentiment,nvi,pacing_zone
0,1,1.63,1,500,0.118,18.1,Low tension / stable emotional tone
1,2,3.26,251,750,0.261,18.7,Low tension / stable emotional tone
2,3,4.89,501,1000,0.211,18.7,Low tension / stable emotional tone
3,4,6.51,751,1250,0.106,19.4,Low tension / stable emotional tone
4,5,8.14,1001,1500,0.091,33.4,Moderate pacing
5,6,9.77,1251,1750,0.235,36.4,Moderate pacing
6,7,11.40,1501,2000,-0.101,35.5,Moderate pacing
7,8,13.03,1751,2250,-0.099,35.3,Moderate pacing
8,9,14.66,2001,2500,0.049,28.9,Moderate pacing
9,10,16.28,2251,2750,0.083,23.9,Moderate pacing


In [13]:
most_volatile = df.sort_values("nvi", ascending=False).head(10).copy()

most_volatile["sentiment"] = most_volatile["sentiment"].round(3)
most_volatile["nvi"] = most_volatile["nvi"].round(1)

most_volatile[[
    "chunk_id",
    "progress_pct",
    "start_word",
    "end_word",
    "sentiment",
    "nvi",
    "pacing_zone"
]]

,chunk_id,progress_pct,start_word,end_word,sentiment,nvi,pacing_zone
48,49,79.79,12001,12500,0.239,100.0,High tension / rapid emotional shifts
47,48,78.17,11751,12250,-0.059,92.1,High tension / rapid emotional shifts
49,50,81.42,12251,12750,0.499,61.6,High tension / rapid emotional shifts
46,47,76.54,11501,12000,-0.392,61.5,High tension / rapid emotional shifts
26,27,43.97,6501,7000,0.157,52.7,High tension / rapid emotional shifts
44,45,73.28,11001,11500,-0.235,49.3,High tension / rapid emotional shifts
30,31,50.48,7501,8000,0.135,43.1,High tension / rapid emotional shifts
57,58,94.45,14251,14750,0.443,42.7,High tension / rapid emotional shifts
29,30,48.85,7251,7750,0.236,41.7,High tension / rapid emotional shifts
22,23,37.45,5501,6000,0.133,39.8,High tension / rapid emotional shifts


In [14]:
calmest_sections = df.sort_values("nvi", ascending=True).head(10).copy()

calmest_sections["sentiment"] = calmest_sections["sentiment"].round(3)
calmest_sections["nvi"] = calmest_sections["nvi"].round(1)

calmest_sections[[
    "chunk_id",
    "progress_pct",
    "start_word",
    "end_word",
    "sentiment",
    "nvi",
    "pacing_zone"
]]

,chunk_id,progress_pct,start_word,end_word,sentiment,nvi,pacing_zone
54,55,89.56,13501,14000,0.546,14.2,Low tension / stable emotional tone
34,35,57.00,8501,9000,0.003,15.7,Low tension / stable emotional tone
55,56,91.19,13751,14250,0.582,16.9,Low tension / stable emotional tone
0,1,1.63,1,500,0.118,18.1,Low tension / stable emotional tone
2,3,4.89,501,1000,0.211,18.7,Low tension / stable emotional tone
1,2,3.26,251,750,0.261,18.7,Low tension / stable emotional tone
17,18,29.31,4251,4750,0.119,18.7,Low tension / stable emotional tone
38,39,63.51,9501,10000,0.162,18.7,Low tension / stable emotional tone
33,34,55.37,8251,8750,0.111,18.9,Low tension / stable emotional tone
3,4,6.51,751,1250,0.106,19.4,Low tension / stable emotional tone


In [15]:
def interpret_pacing(row):
    if row["nvi"] >= high_threshold:
        return "This section has high emotional movement. It may contain conflict, surprise, reversal, or rapid tonal change."
    elif row["nvi"] <= low_threshold:
        return "This section has a stable emotional tone. It may be calm world-building, but it could feel flat if it lasts too long."
    else:
        return "This section has moderate pacing. The emotional movement is present but not extreme."


def suggest_revision(row):
    if row["nvi"] >= high_threshold:
        return "Check whether the reader has enough context to understand the emotional shift. High tension works best when the stakes are clear."
    elif row["nvi"] <= low_threshold:
        return "Consider adding a small change: a new question, a character reaction, a decision, a discovery, or a shift in mood."
    else:
        return "This section is balanced. Look at whether it supports the surrounding high and low tension moments."


df["plain_english_interpretation"] = df.apply(interpret_pacing, axis=1)
df["writing_revision_suggestion"] = df.apply(suggest_revision, axis=1)

writer_feedback = df[[
    "chunk_id",
    "progress_pct",
    "sentiment",
    "nvi",
    "pacing_zone",
    "plain_english_interpretation",
    "writing_revision_suggestion"
]].copy()

writer_feedback["sentiment"] = writer_feedback["sentiment"].round(3)
writer_feedback["nvi"] = writer_feedback["nvi"].round(1)

writer_feedback.head(12)

,chunk_id,progress_pct,sentiment,nvi,pacing_zone,plain_english_interpretation,writing_revision_suggestion
0,1,1.63,0.118,18.1,Low tension / stable emotional tone,This section has a stable emotional tone. It m...,Consider adding a small change: a new question...
1,2,3.26,0.261,18.7,Low tension / stable emotional tone,This section has a stable emotional tone. It m...,Consider adding a small change: a new question...
2,3,4.89,0.211,18.7,Low tension / stable emotional tone,This section has a stable emotional tone. It m...,Consider adding a small change: a new question...
3,4,6.51,0.106,19.4,Low tension / stable emotional tone,This section has a stable emotional tone. It m...,Consider adding a small change: a new question...
4,5,8.14,0.091,33.4,Moderate pacing,This section has moderate pacing. The emotiona...,This section is balanced. Look at whether it s...
5,6,9.77,0.235,36.4,Moderate pacing,This section has moderate pacing. The emotiona...,This section is balanced. Look at whether it s...
6,7,11.40,-0.101,35.5,Moderate pacing,This section has moderate pacing. The emotiona...,This section is balanced. Look at whether it s...
7,8,13.03,-0.099,35.3,Moderate pacing,This section has moderate pacing. The emotiona...,This section is balanced. Look at whether it s...
8,9,14.66,0.049,28.9,Moderate pacing,This section has moderate pacing. The emotiona...,This section is balanced. Look at whether it s...
9,10,16.28,0.083,23.9,Moderate pacing,This section has moderate pacing. The emotiona...,This section is balanced. Look at whether it s...


In [16]:
summary = {
    "Total chunks analyzed": len(df),
    "Average sentiment": round(df["sentiment"].mean(), 3),
    "Sentiment range": round(df["sentiment"].max() - df["sentiment"].min(), 3),
    "Average Narrative Volatility Index": round(df["nvi"].mean(), 1),
    "Highest NVI": round(df["nvi"].max(), 1),
    "Lowest NVI": round(df["nvi"].min(), 1),
    "Most volatile chunk": int(df.loc[df["nvi"].idxmax(), "chunk_id"]),
    "Calmest chunk": int(df.loc[df["nvi"].idxmin(), "chunk_id"])
}

summary_df = pd.DataFrame(summary.items(), columns=["Metric", "Value"])
summary_df

,Metric,Value
0,Total chunks analyzed,60.000
1,Average sentiment,0.136
2,Sentiment range,0.987
3,Average Narrative Volatility Index,33.100
4,Highest NVI,100.000
5,Lowest NVI,14.200
6,Most volatile chunk,49.000
7,Calmest chunk,55.000


In [17]:
output_file = "narrative_volatility_results.csv"

writer_feedback.to_csv(output_file, index=False)

print(f"Saved results to {output_file}")

Saved results to narrative_volatility_results.csv


## ISB NWC 

Middle-school writers(and writers in general) often receive feedback such as "add more tension" or "the pacing feels flat," but those comments can feel vague. The proposed index aims to quantify narrative feedback and provide it in a more constructive manner.

A student writer could upload a draft, split it into rolling sections, and receive a pacing report showing:
- where the emotional tone stays flat for too long
- where the story changes too suddenly
- where the highest-tension moments occur
- whether the climax appears too early, too late, or not clearly enough
- whether the ending creates emotional resolution

This does not replace the provided feedback. Rather, it gives young writers a visual map of their story so they can revise with more intention.

In [18]:
def analyze_novel_file(path, chunk_size=500, step_size=250, volatility_window=5):
    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        raw = file.read()

    text = strip_gutenberg_boilerplate(raw)
    tokens = re.findall(r"\S+", text)

    rows = []

    for start in range(0, len(tokens) - chunk_size + 1, step_size):
        chunk_tokens = tokens[start:start + chunk_size]
        chunk_text = " ".join(chunk_tokens)

        rows.append({
            "chunk_id": len(rows) + 1,
            "start_word": start + 1,
            "end_word": start + len(chunk_tokens),
            "progress_pct": round(((start + chunk_size / 2) / len(tokens)) * 100, 2),
            "text": chunk_text
        })

    novel_df = pd.DataFrame(rows)

    if len(novel_df) == 0:
        return None

    novel_df["sentiment"] = novel_df["text"].apply(chunk_sentiment_score)

    novel_df["nvi_raw"] = novel_df["sentiment"].rolling(
        window=volatility_window,
        center=True,
        min_periods=2
    ).std()

    max_vol = novel_df["nvi_raw"].max()

    if pd.isna(max_vol) or max_vol == 0:
        novel_df["nvi"] = 0
    else:
        novel_df["nvi"] = (novel_df["nvi_raw"] / max_vol) * 100

    novel_name = os.path.basename(path).replace(".txt", "").replace("_", " ").title()
    novel_df["novel"] = novel_name

    return novel_df

In [19]:
all_novel_dfs = []

for path in txt_files:
    result = analyze_novel_file(path)

    if result is not None:
        all_novel_dfs.append(result)

combined_df = pd.concat(all_novel_dfs, ignore_index=True)

combined_df[["novel", "chunk_id", "progress_pct", "sentiment", "nvi"]].head()

,novel,chunk_id,progress_pct,sentiment,nvi
0,Alice In Wonderland,1,1.63,0.118188,18.147016
1,Alice In Wonderland,2,3.26,0.260592,18.702495
2,Alice In Wonderland,3,4.89,0.211045,18.677046
3,Alice In Wonderland,4,6.51,0.105683,19.401276
4,Alice In Wonderland,5,8.14,0.091033,33.364948


In [20]:
fig = px.line(
    combined_df,
    x="progress_pct",
    y="nvi",
    color="novel",
    title="Comparing Narrative Volatility Across Novels",
    labels={
        "progress_pct": "Novel Progress (%)",
        "nvi": "Narrative Volatility Index",
        "novel": "Novel"
    }
)

fig.update_layout(
    height=550,
    template="plotly_white"
)

fig.show()

In [21]:
novel_fingerprints = combined_df.groupby("novel").agg(
    average_sentiment=("sentiment", "mean"),
    sentiment_range=("sentiment", lambda x: x.max() - x.min()),
    average_nvi=("nvi", "mean"),
    max_nvi=("nvi", "max"),
    low_tension_ratio=("nvi", lambda x: (x <= x.quantile(0.20)).mean()),
    high_tension_ratio=("nvi", lambda x: (x >= x.quantile(0.80)).mean())
).reset_index()

novel_fingerprints = novel_fingerprints.round(3)

novel_fingerprints

,novel,average_sentiment,sentiment_range,average_nvi,max_nvi,low_tension_ratio,high_tension_ratio
0,Alice In Wonderland,0.136,0.987,33.069,100.0,0.200,0.200
1,Frankenstein,0.057,1.003,40.996,100.0,0.200,0.200
2,Pride And Prejudice,0.319,0.000,0.000,0.0,1.000,1.000
3,The Adventures Of Sherlock Holmes,0.054,0.653,33.353,100.0,0.201,0.201


## Conclusion

By splitting a novel into rolling 500-word chunks, scoring sentiment, and calculating rolling standard deviation, this notebook transforms a story into a measurable emotional signal. High NVI sections often represent moments of rapid emotional movement, while low NVI sections represent stability, exposition, or possible pacing flatness.

The most important insight is that volatility is not automatically good or bad. A strong story needs contrast. Calm sections can give readers time to understand characters and settings. High-volatility sections can create tension, conflict, or surprise.

For the ISB Novel Writing Challenge, this tool will help middle-school writers revise their drafts by showing them where their stories may need more tension, more clarity, or more breathing room.